# Full-stack pseudo-labelling

This notebook evaluates whether pseudo-labelled transactions can improve the chronological HGB, five-seed CatBoost and LightGBM stack. The pseudo-label policy was chosen in an earlier screening experiment and is held fixed here. The same stack supplies the teacher probabilities; a separate HGB model is trained with selected pseudo-labels.

The policy selects transactions with teacher probabilities below 0.02 or above 0.98, caps the number selected per class at 10% of the historical labelled sample size, and assigns pseudo-labelled rows a weight of 0.1. Final predictions combine 90% of the teacher score with 10% of the student score.

The evaluation includes two 14-step forecasting windows and two shorter windows. These periods were used during earlier model development, so the results are not independent confirmation of the policy. The evaluation labels are used only for scoring, not for selecting pseudo-labels or fitting models.


In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
DATA=ROOT/'data'/'raw'
train=pd.read_csv(DATA/'train.csv')
test=pd.read_csv(DATA/'test.csv')
sample=pd.read_csv(DATA/'sample_submission.csv')
labelled=train.dropna(subset=['label']).copy()
features=[c for c in train.columns if c.startswith('feat_')]
assert {'txId','time_step','label'}.issubset(train.columns)
assert labelled.txId.is_unique and labelled.label.isin([0,1]).all()
assert all(c in test for c in features)
assert list(sample.columns)==['index','target']
assert len(sample)==len(test)
print('Labelled:',len(labelled),'Test:',len(test),'Features:',len(features))

Labelled: 31235 Test: 15329 Features: 165


In [2]:
SEEDS=[11,42,73,101,202]
HGB_PARAMS=dict(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,l2_regularization=1.0,random_state=42)
CAT_PARAMS=dict(iterations=700,depth=6,learning_rate=0.045,l2_leaf_reg=5,loss_function='Logloss',verbose=False,thread_count=-1)
LGB_PARAMS=dict(n_estimators=700,learning_rate=0.045,num_leaves=31,max_depth=-1,reg_lambda=5.0,random_state=42,verbosity=-1,n_jobs=-1)
OOF_WINDOWS=[(7,9),(10,12),(13,14),(15,17),(18,20),(21,21),(22,24),(25,29),(30,35)]
EVALUATIONS=[('long14',14,15,28),('long21',21,22,35),('recent25',24,25,29),('recent30',29,30,35)]
LOW,HIGH=0.02,0.98
PSEUDO_WEIGHT=0.1
STUDENT_BLEND=0.1
CLASS_CAP_FRACTION=0.10
CACHE=ROOT/'outputs'/'experiment25_cache'
CACHE.mkdir(parents=True,exist_ok=True)
config=repr((SEEDS,HGB_PARAMS,CAT_PARAMS,LGB_PARAMS,features))
data_hash=pd.util.hash_pandas_object(labelled[['txId','time_step','label']+features],index=True).values
fingerprint=hashlib.sha256(config.encode()+data_hash.tobytes()).hexdigest()[:16]
print('Cache fingerprint:',fingerprint)

def base_predictions(tr,va):
    Xtr=tr[features];ytr=tr.label.astype(int);Xva=va[features]
    if ytr.nunique()!=2:raise ValueError('Only one class in training window')
    ph=HistGradientBoostingClassifier(**HGB_PARAMS).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    pc=np.mean([CatBoostClassifier(**CAT_PARAMS,random_seed=s).fit(Xtr,ytr).predict_proba(Xva)[:,1] for s in SEEDS],axis=0)
    pl=LGBMClassifier(**LGB_PARAMS).fit(Xtr,ytr).predict_proba(Xva)[:,1]
    return np.column_stack([ph,pc,pl])

def window_predictions(lo,hi,train_end):
    assert train_end<lo
    tr=labelled[labelled.time_step<=train_end]
    va=labelled[labelled.time_step.between(lo,hi)].copy()
    if tr.empty or va.empty:raise ValueError(f'Empty training/evaluation slice: {train_end} -> {lo}-{hi}')
    key=f'{fingerprint}_tr{train_end}_eval{lo}_{hi}.npz'
    path=CACHE/key
    ids=np.asarray(va.txId.astype(str).tolist(),dtype=str)
    y=va.label.to_numpy(dtype=int)
    if path.exists():
        try:
            with np.load(path,allow_pickle=False) as z:
                p=z['p'];cached_ids=z['ids'];cached_y=z['y']
            if not(np.array_equal(ids,cached_ids) and np.array_equal(y,cached_y) and p.shape==(len(y),3)):
                raise ValueError('Stale cache')
            print('Loaded:',key)
            return va,p
        except (ValueError,KeyError) as exc:
            print('Rebuilding cache:',type(exc).__name__)
    print(f'Training base models <= {train_end}; predicting {lo}-{hi}',flush=True)
    p=base_predictions(tr,va)
    np.savez_compressed(path,ids=ids,y=y,p=p)
    return va,p

def fit_meta(origin):
    parts=[];targets=[];used=[]
    for lo,hi in OOF_WINDOWS:
        if hi>origin:continue
        va,p=window_predictions(lo,hi,lo-1)
        parts.append(p);targets.append(va.label.to_numpy(dtype=int));used.append((lo,hi))
    if not parts:raise ValueError('No OOF windows')
    meta=LogisticRegression(C=0.1,max_iter=2000,solver='lbfgs').fit(np.vstack(parts),np.concatenate(targets))
    return meta,used,sum(len(y) for y in targets)

def select_pseudo(prob,n_train):
    cap=max(1,int(n_train*CLASS_CAP_FRACTION))
    neg=np.flatnonzero(prob<=LOW)
    pos=np.flatnonzero(prob>=HIGH)
    neg=neg[np.argsort(prob[neg],kind='stable')[:cap]]
    pos=pos[np.argsort(-prob[pos],kind='stable')[:cap]]
    ix=np.sort(np.concatenate([neg,pos]))
    return ix,(prob[ix]>=HIGH).astype(int),len(neg),len(pos)

def student_blend(historical,future,teacher):
    ix,yp,nn,np_=select_pseudo(teacher,len(historical))
    if len(ix)==0:
        return teacher.copy(),dict(selected=0,negative=0,positive=0)
    X=pd.concat([historical[features],future.iloc[ix][features]],ignore_index=True)
    y=np.concatenate([historical.label.to_numpy(dtype=int),yp])
    weights=np.concatenate([np.ones(len(historical)),np.full(len(ix),PSEUDO_WEIGHT)])
    student=HistGradientBoostingClassifier(**HGB_PARAMS).fit(X,y,sample_weight=weights)
    ps=student.predict_proba(future[features])[:,1]
    return (1-STUDENT_BLEND)*teacher+STUDENT_BLEND*ps,dict(selected=len(ix),negative=nn,positive=np_)

def metrics(y,p):
    if len(np.unique(y))<2:return np.nan,np.nan
    return average_precision_score(y,p),roc_auc_score(y,p)

Cache fingerprint: 4f838a69340df870


## Chronological validation

For each validation period, the base models train on transactions available before the forecast origin. The logistic regression meta-model is fitted on earlier chronological out-of-fold predictions. The resulting teacher scores are used to select pseudo-labels from the unlabelled features in the evaluation period, after which the HGB student is fitted.

This is a transductive approach: the student uses the evaluation transactions' features, but not their true labels. We compare the teacher and blended predictions using PR-AUC and ROC-AUC, including separate checks on the later halves of the long-horizon windows.


In [3]:
rows=[];checks=[];predictions={}
for name,origin,lo,hi in EVALUATIONS:
    print('\n===',name,'origin',origin,'eval',lo,hi,'===',flush=True)
    meta,windows,n_oof=fit_meta(origin)
    future,P=window_predictions(lo,hi,origin)
    teacher=meta.predict_proba(P)[:,1]
    historical=labelled[labelled.time_step<=origin]
    challenger,counts=student_blend(historical,future,teacher)
    predictions[name]=(future,teacher,challenger)
    checks.append(dict(window=name,origin=origin,meta_oof_rows=n_oof,meta_windows=str(windows),**counts))
    y=future.label.to_numpy(dtype=int)
    phases=[('full',np.ones(len(future),dtype=bool))]
    if name.startswith('long'):
        phases.extend([('early',future.time_step.between(origin+1,origin+7).to_numpy()),
                       ('late',future.time_step.between(origin+8,origin+14).to_numpy())])
    for phase,mask in phases:
        bpr,broc=metrics(y[mask],teacher[mask]);cpr,croc=metrics(y[mask],challenger[mask])
        rows.append(dict(window=name,phase=phase,n=int(mask.sum()),prevalence=float(y[mask].mean()),
                         baseline_pr=bpr,pseudo_pr=cpr,delta_pr=cpr-bpr,
                         baseline_roc=broc,pseudo_roc=croc,delta_roc=croc-broc))
results=pd.DataFrame(rows)
coverage=pd.DataFrame(checks)
display(coverage)
display(results.round(6))


=== long14 origin 14 eval 15 28 ===
Training base models <= 6; predicting 7-9
Training base models <= 9; predicting 10-12
Training base models <= 12; predicting 13-14
Training base models <= 14; predicting 15-28

=== long21 origin 21 eval 22 35 ===
Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Training base models <= 14; predicting 15-17
Training base models <= 17; predicting 18-20
Training base models <= 20; predicting 21-21
Training base models <= 21; predicting 22-35

=== recent25 origin 24 eval 25 29 ===
Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Loaded: 4f838a69340df870_tr14_eval15_17.npz
Loaded: 4f838a69340df870_tr17_eval18_20.npz
Loaded: 4f838a69340df870_tr20_eval21_21.npz
Training base models <= 21; predicting 22-24
Training base models <= 24; predicting 25-29

=== recent30 origin 29 eval 30 35 ===
Loaded: 4f838a

,window,origin,meta_oof_rows,meta_windows,selected,negative,positive
0,long14,14,6546,"[(7, 9), (10, 12), (13, 14)]",841,0,841
1,long21,21,11180,"[(7, 9), (10, 12), (13, 14), (15, 17), (18, 20...",1145,0,1145
2,recent25,24,15256,"[(7, 9), (10, 12), (13, 14), (15, 17), (18, 20...",552,0,552
3,recent30,29,18031,"[(7, 9), (10, 12), (13, 14), (15, 17), (18, 20...",613,0,613


,window,phase,n,prevalence,baseline_pr,pseudo_pr,delta_pr,baseline_roc,pseudo_roc,delta_roc
0,long14,full,10311,0.149064,0.921846,0.922394,0.000547,0.956369,0.958385,0.002016
1,long14,early,4634,0.186880,0.980313,0.980265,-0.000048,0.992303,0.992235,-0.000068
2,long14,late,5677,0.118196,0.836568,0.837635,0.001066,0.913673,0.918395,0.004722
3,long21,full,11705,0.151474,0.803385,0.809310,0.005925,0.882974,0.896909,0.013935
4,long21,early,5677,0.118196,0.877244,0.879765,0.002521,0.946175,0.951576,0.005401
5,long21,late,6028,0.182814,0.757596,0.766545,0.008949,0.821307,0.843874,0.022566
6,recent25,full,2775,0.234955,0.993671,0.993759,0.000088,0.997787,0.997630,-0.000157
7,recent30,full,4854,0.159250,0.992336,0.992309,-0.000027,0.998309,0.998240,-0.000068


In [4]:
full=results[results.phase=='full'].copy()
late=results[results.phase=='late'].copy()
long=full[full.window.str.startswith('long')]
recent=full[full.window.str.startswith('recent')]
print('Long-horizon full-window gains:')
display(long[['window','baseline_pr','pseudo_pr','delta_pr','delta_roc']].round(6))
print('Long-horizon late-half gains:')
display(late[['window','delta_pr','delta_roc']].round(6))
print('Recent-window gains (already reused in prior experiments):')
display(recent[['window','delta_pr','delta_roc']].round(6))
LONG_OK=(len(long)==2 and (long.delta_pr>0).all() and len(late)==2 and (late.delta_pr>=-0.0005).all())
RECENT_OK=(len(recent)==2 and (recent.delta_pr>=-0.0005).all())
print('Long-horizon screen passes:',LONG_OK)
print('Recent robustness screen passes:',RECENT_OK)
print('Candidate merits manual review:',bool(LONG_OK and RECENT_OK))
print('Caution: these validation windows are NOT independent of prior experiment choices.')

Long-horizon full-window gains:


,window,baseline_pr,pseudo_pr,delta_pr,delta_roc
0,long14,0.921846,0.922394,0.000547,0.002016
3,long21,0.803385,0.809310,0.005925,0.013935


Long-horizon late-half gains:


,window,delta_pr,delta_roc
2,long14,0.001066,0.004722
5,long21,0.008949,0.022566


Recent-window gains (already reused in prior experiments):


,window,delta_pr,delta_roc
6,recent25,0.000088,-0.000157
7,recent30,-0.000027,-0.000068


Long-horizon screen passes: True
Recent robustness screen passes: True
Candidate merits manual review: True
Caution: these validation windows are NOT independent of prior experiment choices.


In [6]:
audit = []

for name, (future, teacher, challenger) in predictions.items():
    ix, pseudo_y, n_neg, n_pos = select_pseudo(
        teacher,
        len(labelled[labelled.time_step <= dict(
            (n, origin) for n, origin, _, _ in EVALUATIONS
        )[name]])
    )

    actual_y = future.iloc[ix]["label"].to_numpy(dtype=int)

    audit.append({
        "window": name,
        "selected": len(ix),
        "positive_precision": (
            actual_y.mean() if len(ix) else np.nan
        ),
        "mean_teacher_probability": (
            teacher[ix].mean() if len(ix) else np.nan
        ),
        "overall_fraud_rate": future["label"].mean()
    })

display(pd.DataFrame(audit).round(5))

,window,selected,positive_precision,mean_teacher_probability,overall_fraud_rate
0,long14,841,0.99643,0.99260,0.14906
1,long21,1145,0.98690,0.99591,0.15147
2,recent25,552,0.99457,0.99240,0.23495
3,recent30,613,0.99511,0.99401,0.15925


## Kaggle submission

The submission code fits the stack using labelled transactions through time step 35 and produces predictions for the unlabelled Kaggle test transactions. It then selects pseudo-labels from the test features and fits the HGB student using the same fixed policy.

This approach assumes that using unlabelled test features for adaptation is permitted by the competition rules. 


In [7]:
MAKE_SUBMISSION=True
if MAKE_SUBMISSION:
    if not (LONG_OK and RECENT_OK):
        raise RuntimeError('Validation screening did not pass. No submission generated.')
    # Ensure the Kaggle test is chronologically later than all labelled training observations.
    if not (test.time_step.min()>labelled.time_step.max()):
        raise RuntimeError('Unexpected time ordering: verify test time_step range.')
    final_origin=int(labelled.time_step.max())
    final_meta,used,n_oof=fit_meta(final_origin)
    print('Final meta OOF rows:',n_oof,'windows:',used)
    Ptest=base_predictions(labelled,test)
    teacher=final_meta.predict_proba(Ptest)[:,1]
    challenger,counts=student_blend(labelled,test,teacher)
    print('Final pseudo-label selection:',counts)
    if counts['selected']==0:raise RuntimeError('No confident test pseudo-labels; refusing unchanged submission')
    if not np.isfinite(challenger).all() or not np.all((challenger>=0)&(challenger<=1)):
        raise RuntimeError('Invalid prediction values')
    submission=sample.copy()
    submission['target']=challenger
    output=ROOT/'submissions'/'experiment25_pseudo_stack.csv'
    output.parent.mkdir(parents=True,exist_ok=True)
    if output.exists():raise FileExistsError(f'Refusing to overwrite {output}')
    submission.to_csv(output,index=False)
    print('Saved:',output)
else:
    print('Submission disabled. Keep Experiment 18 (public score 0.94914).')

Loaded: 4f838a69340df870_tr6_eval7_9.npz
Loaded: 4f838a69340df870_tr9_eval10_12.npz
Loaded: 4f838a69340df870_tr12_eval13_14.npz
Loaded: 4f838a69340df870_tr14_eval15_17.npz
Loaded: 4f838a69340df870_tr17_eval18_20.npz
Loaded: 4f838a69340df870_tr20_eval21_21.npz
Loaded: 4f838a69340df870_tr21_eval22_24.npz
Loaded: 4f838a69340df870_tr24_eval25_29.npz
Loaded: 4f838a69340df870_tr29_eval30_35.npz
Final meta OOF rows: 22885 windows: [(7, 9), (10, 12), (13, 14), (15, 17), (18, 20), (21, 21), (22, 24), (25, 29), (30, 35)]
Final pseudo-label selection: {'selected': 3686, 'negative': 3123, 'positive': 563}
Saved: submissions/experiment25_pseudo_stack.csv


## Results and interpretation

Pseudo-labelling improved PR-AUC from 0.921846 to 0.922394 on steps 15–28 and from 0.803385 to 0.809310 on steps 22–35. The change was negligible on steps 25–29 (+0.000088) and slightly negative on steps 30–35 (−0.000027).

All selected pseudo-labels in the four historical windows were positive. Their observed precision ranged from 98.690% to 99.643%, but this does not guarantee that pseudo-labels on the later Kaggle test period are equally reliable.

The final Kaggle submission scored 0.94703 publicly, below the original chronological stack's 0.94914. The historical gains therefore did not translate into an improvement on the public test subset. The final meta-model also uses additional out-of-fold windows, so this comparison does not isolate pseudo-labelling as the only difference. The original stack was retained as the preferred submission.
